# 🐍 Day 35 · Python Basics — JSONL Datasets & Train/Test Split (Fine-Tuning)

> **Goal of this notebook:** Fine-tuning needs a clean dataset of example conversations. Learn the JSONL format, shuffling, splitting and the chat template used by Gemma.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. What JSONL is
2. Writing and reading a JSONL file
3. Checking data quality (word limits)
4. Shuffling and splitting into train / test
5. Formatting examples with a chat template
6. Counting and summarising the dataset

## 1. JSONL = one JSON object per line
`dataset.jsonl` in this folder looks like this:
```
{"instruction": "...", "response": "..."}
{"instruction": "...", "response": "..."}
```
It is easy to add lines and to read huge files one line at a time.

In [ ]:
examples = [
    {"instruction": "Translate 'water' to Hindi", "response": "पानी (paani)"},
    {"instruction": "Translate 'book' to Hindi", "response": "किताब (kitaab)"},
    {"instruction": "Translate 'friend' to Hindi", "response": "दोस्त (dost)"},
    {"instruction": "Translate 'house' to Hindi", "response": "घर (ghar)"},
    {"instruction": "Translate 'sun' to Hindi", "response": "सूरज (sooraj)"},
    {"instruction": "Translate 'tree' to Hindi", "response": "पेड़ (ped)"},
    {"instruction": "Translate 'school' to Hindi", "response": "स्कूल (school)"},
    {"instruction": "Translate 'food' to Hindi", "response": "खाना (khaana)"},
    {"instruction": "Translate 'night' to Hindi", "response": "रात (raat)"},
    {"instruction": "Translate 'mother' to Hindi", "response": "माँ (maa)"},
]
print(len(examples), "examples")

## 2. Write and read JSONL
`ensure_ascii=False` keeps Hindi letters readable instead of `प...` codes.

In [ ]:
import json

with open("demo_dataset.jsonl", "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")     # one object per line

loaded = []
with open("demo_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        loaded.append(json.loads(line))

print(loaded[0])
print(open("demo_dataset.jsonl", encoding="utf-8").read()[:150])

## 3. Quality checks
Bad training data = bad model. `generate_dataset.py` keeps every response under a word limit.

In [ ]:
MAX_WORDS = 300

def is_valid(ex):
    return bool(ex["instruction"].strip()) and bool(ex["response"].strip()) and len(ex["response"].split()) <= MAX_WORDS

bad_example = {"instruction": "", "response": "?"}
print(is_valid(examples[0]), is_valid(bad_example))
clean = [ex for ex in loaded if is_valid(ex)]
print(len(clean), "valid examples")

## 4. Shuffle and split
The model learns from the **train** set; we check it on the **test** set it has never seen. `random.seed` makes the shuffle repeatable.

In [ ]:
import random

random.seed(42)
data = clean.copy()
random.shuffle(data)

split = int(len(data) * 0.8)          # 80% train, 20% test
train, test = data[:split], data[split:]
print(len(train), "train |", len(test), "test")
print("Test words:", [ex["instruction"].split("'")[1] for ex in test])

## 5. 🔗 The chat template
Gemma expects every example in its special **chat format**. `train.py` formats the data like this before training.

In [ ]:
def to_gemma_chat(ex):
    return (
        "<start_of_turn>user\n" + ex["instruction"] + "<end_of_turn>\n"
        "<start_of_turn>model\n" + ex["response"] + "<end_of_turn>"
    )

print(to_gemma_chat(train[0]))

## 6. Summarise the dataset

In [ ]:
from statistics import mean
lengths = [len(to_gemma_chat(ex)) for ex in train]
print(f"Train examples: {len(train)}, average length: {mean(lengths):.0f} characters, longest: {max(lengths)}")

import os
os.remove("demo_dataset.jsonl")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add two more examples (any words you like) and rebuild `clean`.
2. Change the split to 70 / 30.
3. Write `count_words(ex)` that counts the words of instruction + response.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
examples.append({"instruction": "Translate 'rain' to Hindi", "response": "बारिश (baarish)"})
examples.append({"instruction": "Translate 'city' to Hindi", "response": "शहर (shahar)"})

split = int(len(data) * 0.7)

def count_words(ex):
    return len((ex["instruction"] + " " + ex["response"]).split())
print(count_words(examples[0]))
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Write JSONL | `f.write(json.dumps(ex, ensure_ascii=False) + '\n')` |
| Read JSONL | `[json.loads(line) for line in f]` |
| Repeatable shuffle | `random.seed(42); random.shuffle(data)` |
| Split | `train, test = data[:split], data[split:]` |
| Chat template | `'<start_of_turn>user\n' + text` |
| Average | `statistics.mean(lengths)` |

## ➡️ What's next?
You now know the Python used in **Day_35_Fine_Tuning_LLMs**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀